# Notebook 4: Model Evaluation and Visualization
- Load best model and validation dataset
- Compute metrics (accuracy, precision, recall, F1)
- Generate and plot confusion matrix and ROC curves per class
- Visualize model interpretability with Grad-CAM on sample images
- Allow user to upload custom images for prediction and visualization

In [ ]:
# Import libraries
import os
import io
import cv2
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from ipywidgets import FileUpload
from IPython.display import display

# Set device to GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the 7 skin lesion classes
class_names = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']

# Load the trained ResNet18 model
from torchvision.models import resnet18, ResNet18_Weights

# Load the base ResNet18 model with pretrained ImageNet weights
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Replace the final layer to output 7 classes instead of 1000
model.fc = torch.nn.Linear(model.fc.in_features, len(class_names))

# Load the saved trained model weights
model.load_state_dict(torch.load("best_resnet18_model.pth", map_location=device))
model = model.to(device)
model.eval()

# Define dataset and dataloader for validation
val_df = pd.read_csv("val_df_processed.csv")

# Custom dataset class to read image paths and labels from the DataFrame
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        image_path = self.data.iloc[idx]['image_path']
        label = self.data.iloc[idx]['label_idx']
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# Image preprocessing: resize, convert to tensor, and normalize
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Create a DataLoader for the validation dataset
val_loader = DataLoader(SkinLesionDataset(val_df, transform=preprocess),
                        batch_size=16, shuffle=False)

# Model evaluation and metrics calculation
def evaluate_model(model, val_loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
    return y_true, y_pred

# Evaluate the model and get predictions
y_true, y_pred = evaluate_model(model, val_loader)

# Compute and print accuracy, precision, recall, and F1-score
metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, average='weighted', zero_division=0),
    "recall": recall_score(y_true, y_pred, average='weighted', zero_division=0),
    "f1_score": f1_score(y_true, y_pred, average='weighted', zero_division=0)
}

for k, v in metrics.items():
    print(f"{k.capitalize()}: {v:.4f}")

# Save metrics to CSV
pd.DataFrame(metrics.items(), columns=["Metric", "Value"]).to_csv("metrics_summary.csv", index=False)

# Confusion matrix visualization
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# ROC curves for each class
# First, get prediction probabilities for all samples
model.eval()
all_probs, all_labels = [], []
softmax = torch.nn.Softmax(dim=1)

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs = softmax(outputs).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

# Stack all results and one-hot encode the labels
all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
all_labels_onehot = label_binarize(all_labels, classes=range(len(class_names)))

# Plot ROC curve for each class
plt.figure(figsize=(10, 8))
for i, class_name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(all_labels_onehot[:, i], all_probs[:, i])
    auc_score = roc_auc_score(all_labels_onehot[:, i], all_probs[:, i])
    plt.plot(fpr, tpr, label=f"{class_name} (AUC = {auc_score:.2f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Per-Class ROC Curves")
plt.legend(loc='lower right')
plt.show()

# Grad-CAM: visual explanation of model predictions
output_folder = "gradcam_outputs"
os.makedirs(output_folder, exist_ok=True)

# Function to generate Grad-CAM for a single image
def generate_gradcam(image_path, model, target_layer):
    rgb_img = Image.open(image_path).convert("RGB")
    input_tensor = preprocess(rgb_img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        pred_class = probs.argmax(dim=1).item()

    cam = GradCAM(model=model, target_layers=[target_layer])
    targets = [ClassifierOutputTarget(pred_class)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

    rgb_img_np = np.array(rgb_img) / 255.0
    visualization = show_cam_on_image(rgb_img_np, grayscale_cam, use_rgb=True)

    return rgb_img, visualization, pred_class, probs[0, pred_class].item()

# Pick 5 random validation images and show their Grad-CAM
target_layer = model.layer4[-1]
sample_images = val_df.sample(5)['image_path'].values

plt.figure(figsize=(12, 8))
for i, img_path in enumerate(sample_images):
    orig_img, cam_img, pred_cls, conf = generate_gradcam(img_path, model, target_layer)
    
    plt.subplot(2, 5, i+1)
    plt.imshow(orig_img)
    plt.axis("off")
    plt.title("Original")

    plt.subplot(2, 5, i+6)
    plt.imshow(cam_img)
    plt.axis("off")
    plt.title(f"{class_names[pred_cls]}\n({conf:.2f})")

    # Save CAM image
    cam_img_bgr = cv2.cvtColor(cam_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(os.path.join(output_folder, f"{os.path.basename(img_path)}_cam.png"), cam_img_bgr)

plt.tight_layout()
plt.show()

# Allow users to upload their own images and get predictions
upload = FileUpload(accept='image/*', multiple=True)
display(upload)

if upload.value:
    plt.figure(figsize=(15, 5))
    for i, file_info in enumerate(upload.value):
        img = Image.open(io.BytesIO(file_info['content'])).convert("RGB")
        input_tensor = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = model(input_tensor)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            pred_idx = torch.argmax(probs, dim=1).item()
            conf = probs[0, pred_idx].item()

        plt.subplot(1, len(upload.value), i+1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"{class_names[pred_idx]}\n({conf*100:.1f}%)")
    plt.tight_layout()
    plt.show()
else:
    print("Please upload at least one image.")


Evaluating:  72%|███████▏  | 91/126 [00:19<00:07,  4.73it/s]